# Homework 3: The Gender Wage Gap

**Research question:** How do hourly wages differ between women and men,
and does the relationship between education and wages differ between them?

Use a historical sample of workers from the **1985 US Current Population Survey**.
First estimate a model with a female indicator, then add education and potential
experience, and finally allow the education slope to differ between women and men.
These regressions describe associations in this sample.

**What to hand in:** this notebook with working code, saved outputs, and completed
**Answer 1–3** cells. Use your own explanations and the numbers your code produces.
Keep comparisons in log-wage units; no conversion to dollar wages is required.

First resolve execution problems. Before revising the analysis, record the initial
coefficient requested in Answer 2. Then check and repair the analysis, complete
the answers, restart the kernel, run all cells, and save.
Read `README.md` for setup, the deadline, and submission instructions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")

### Question 1 — Get the notebook running

Run the notebook and repair anything that prevents execution. Briefly explain
the error, its cause, your repair, and how you verified it.

### Answer 1

- Error and cause: _Write your answer._
- Repair and verification: _Write your answer._

## 1. Inspect the data

Each row represents one worker. `wage` is hourly pay in US dollars in 1985;
`education` is years of education. `experience` measures **potential** work
experience derived from age and schooling, not a recorded employment history.

The source's `gender` field has the labels `male` and `female`; this exercise
uses those recorded categories. `rownames` is a source row label, not a predictor.
See `data/README.md` for the source and variable definitions.

In [ ]:
workers = pd.read_csv("data/cps1985.csv")
print("Rows:", len(workers))
# Exclude the source row label when checking for identical data records.
print("Exact duplicate records:", workers.drop(columns="rownames").duplicated().sum())
workers.isna().sum().rename("missing")

In [ ]:
workers["Gender"].value_counts()

In [ ]:
required = ["wage", "education", "experience", "gender"]
assert not workers[required].isna().any().any(), "Investigate missing model variables."
assert workers["gender"].isin(["male", "female"]).all(), "Check the category labels."
assert np.isfinite(workers[["wage", "education", "experience"]].to_numpy()).all()
assert workers["wage"].gt(0).all(), "Log wages require positive wages."
assert workers[["education", "experience"]].ge(0).all().all()

# Code the indicator as 1 for female and 0 for male, the reference group.
workers["female"] = workers["gender"].eq("male").astype(int)
workers["log_wage"] = np.log(workers["wage"])
workers[["wage", "education", "experience"]].describe().round(2)

Use the original years-of-education variable throughout. Retain all supplied
observations; do not remove workers solely because their wages are unusual.
All models use the same sample and HC0 robust standard errors.

In [ ]:
sns.boxplot(data=workers, x="gender", y="log_wage", order=["male", "female"])
plt.title("Log hourly wages by gender, 1985 CPS sample")
plt.xlabel("Recorded gender")
plt.ylabel("Log hourly wage")
plt.show()

In [ ]:
# These are unadjusted sample summaries.
workers.groupby("gender").agg(
    workers=("log_wage", "size"),
    mean_log_wage=("log_wage", "mean"),
    mean_education=("education", "mean"),
    mean_experience=("experience", "mean"),
).round(3)

## 2. Estimate the wage difference

Start with the female indicator alone. Then add education and potential
experience as numeric predictors. The second model allows different wage
levels but assumes a common education slope for women and men.

After resolving execution problems, record the initial `female` coefficient
from `adjusted_model` in Answer 2 before making further corrections.

In [ ]:
gender_model = smf.ols(
    "log_wage ~ female", data=workers, missing="raise"
).fit(cov_type="HC0")
gender_model.params.round(6)

In [ ]:
adjusted_model = smf.ols(
    "log_wage ~ female + education + experience",
    data=workers, missing="raise",
).fit(cov_type="HC0")
adjusted_model.summary()

### Question 2 — Check and interpret the wage gap

The `female` coefficient is intended to measure the female–male log-wage
difference. Check whether the indicator used in the regression supports that
interpretation. Show a small table comparing `female` with the recorded
`gender` labels, and correct any inconsistency.

Rerun the models. Report the `female` coefficient in **`adjusted_model`** to
four decimal places and interpret its sign and units. State the reference
group and what is held constant.

### Answer 2

- Initial `female` coefficient in `adjusted_model`: _Record the value before corrections._
- Indicator check and any repair: _Show your table and explain what you found._
- Final `female` coefficient in `adjusted_model`: _Report the value._
- Interpretation, reference group, and what is held constant: _Write your answer._

## 3. Allow different education slopes

Add an interaction between the female indicator and years of education.
`*` includes both predictors and their interaction. Keep potential experience
and the estimation sample unchanged.

With an interaction, a main-effect coefficient is evaluated at the other
interacting variable's zero value. Do not automatically carry the female
coefficient's interpretation from the preceding model into this one.

In [ ]:
interaction_model = smf.ols(
    "log_wage ~ female * education + experience",
    data=workers, missing="raise",
).fit(cov_type="HC0")
assert gender_model.nobs == adjusted_model.nobs == interaction_model.nobs == len(workers)
interaction_model.summary()

The code below calculates the education slopes for you. With the intended
coding (`female = 0` for men and `female = 1` for women), the education
coefficient is the slope for men. Adding the interaction coefficient gives
the slope for women. Each slope measures a change in fitted log wage per
additional education year, holding potential experience constant.

Use the table from your corrected analysis to answer Question 3.

In [ ]:
# Men are the reference group (female = 0).
male_slope = interaction_model.params["education"]
# For women (female = 1), add the interaction to the reference-group slope.
female_slope = male_slope + interaction_model.params["female:education"]

education_slopes = pd.Series({
    "male": male_slope,
    "female": female_slope,
}, name="Log-wage change per additional year of education")
education_slopes.round(6)

### Question 3 — Interpret the education slopes

Use the supplied slope calculations from your corrected analysis. Report the
education slope for men and for women to six decimal places. Interpret each
as the fitted log-wage change associated with one additional education year,
stating what is held constant. Which group has the steeper relationship?

### Answer 3

| Group | Education slope | Interpretation |
| --- | --- | --- |
| male | _Fill in._ | _Write your interpretation._ |
| female | _Fill in._ | _Write your interpretation._ |

**Comparison:** _Which group has the steeper relationship, holding potential
experience constant?_

## Submission checklist

- [ ] Answers 1–3 are complete, including the indicator check in Question 2.
- [ ] The supplied dataset is unchanged.
- [ ] I restarted the kernel, ran all cells without errors, and saved the outputs.
- [ ] I committed my work and pushed it to my own fork, as described in `README.md`.